# Personalized TED Talks Recommendation System
## Milestone 1: Data Collection, Exploration & Preprocessing

**Author:** Ayush Gupta  
**Project:** TED Talks Recommendation Engine

This notebook covers the initial pipeline stages:
1. Loading the TED Talks metadata and verbatim transcripts
2. Cleaning text, resolving missing entries, and extracting speech/duration metrics
3. Exploratory analysis on talk durations, tags, audience reach, and feedback
4. Preparing content soup and user-talk interaction matrices for recommendation models

### 1. Environment Setup

In [1]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

### 2. Inspecting Raw Data
We use two main files:
- `ted_main.csv`: Talk metadata (speakers, occupations, tags, views, duration, ratings)
- `transcripts.csv`: Verbatim spoken transcripts

In [2]:
df_main = pd.read_csv('../data/raw/ted_main.csv', encoding='latin1')
df_trans = pd.read_csv('../data/raw/transcripts.csv', encoding_errors='replace')

print('Metadata rows & cols:', df_main.shape)
print('Transcripts rows & cols:', df_trans.shape)
df_main[['title', 'main_speaker', 'speaker_occupation', 'views', 'duration']].head()

### 3. Data Cleaning and Merging
Key preprocessing steps:
- Merge metadata and transcripts on `url`
- Impute missing transcripts using title and description so all talks can be recommended
- Fill missing speaker occupations with 'Unknown'
- Parse stringified tags and rating dictionaries into native Python structures

In [3]:
df_cleaned = pd.read_csv('../data/processed/ted_talks_cleaned.csv')
print('Cleaned dataset shape:', df_cleaned.shape)
print('Remaining null values:', df_cleaned.isnull().sum().sum())
df_cleaned[['title', 'main_speaker', 'duration_minutes', 'transcript_word_count', 'dominant_sentiment']].head()

### 4. Text Processing and Feature Engineering
For content-based filtering, we need clean text representations:
- Strip stage cues like `(Laughter)` and `(Applause)`
- Lowercase and remove punctuation/symbols
- Filter standard stopwords
- Build a combined `content_soup` containing title, speaker, occupation, tags, and description

In [4]:
sample = df_cleaned.iloc[0]
print('Title:', sample['title'])
print('Tags:', sample['tags_list'])
print('\nCombined Content Soup (first 250 chars):')
print(sample['content_soup'][:250], '...')

### 5. Exploratory Data Analysis
Visualizing talk length distributions and pacing.

In [5]:
plt.figure(figsize=(9, 5))
sns.histplot(df_cleaned['duration_minutes'], bins=40, kde=True, color='#e62b1e')
plt.axvline(df_cleaned['duration_minutes'].mean(), color='#2c3e50', linestyle='--', label=f"Mean: {df_cleaned['duration_minutes'].mean():.1f}m")
plt.axvline(df_cleaned['duration_minutes'].median(), color='#27ae60', linestyle='-', label=f"Median: {df_cleaned['duration_minutes'].median():.1f}m")
plt.title('Talk Duration Distribution (Minutes)')
plt.xlabel('Duration (Minutes)')
plt.legend()
plt.show()

### 6. User-Item Interaction Matrix for Collaborative Filtering
Simulating viewing activity and ratings (1-5 stars) across 500 user profiles with diverse topic affinities for collaborative filtering in Milestone 2.

In [6]:
df_users = pd.read_csv('../data/processed/user_item_interactions.csv')
print(f'Total interactions: {len(df_users):,}')
print(f'Distinct users: {df_users["user_id"].nunique()}')
print(f'Distinct talks rated: {df_users["talk_id"].nunique()}')
df_users.head()

### 7. Summary Statistics

In [7]:
with open('../data/processed/data_summary_stats.json') as f:
    stats = json.load(f)
pd.Series(stats)